In [ ]:
import pandas as pd
import os

import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
import torch
torch.manual_seed(100)

import random
random.seed(15)

import numpy as np
np.random.seed(30)

In [ ]:
with torch.no_grad():
    torch.cuda.empty_cache()

In [ ]:
DIR_INPUTS = './data_raw/'
DIR_RUNTIME_DATA = './data_runtime/'
DIR_RUNTIME_RESULTS = './results_runtime/'
DIR_RESUlTS_END = './results_end/'
TRANS_ONLY=False
g_resol = 1

### Create train-validation and test sets (into csvs)

In [ ]:
MAX_STATES = 5
N_TIME = 10
USE_GPU = False

In [ ]:
MAX_STATES

## Load train val and test sets

In [ ]:
from monotonic_nn_surv_surf.utils.datasets_def import DatasetFeatANDtgy
from torch.utils.data import DataLoader

In [ ]:
ds_train = DatasetFeatANDtgy(
    path_feat_by_subj=os.path.join(DIR_RUNTIME_DATA,'df_features_train.csv'),
    path_state_history_max_grade=os.path.join(DIR_RUNTIME_DATA,'df_state_history_sampled_max_train.csv'),
    max_grade=MAX_STATES, 
    max_time=N_TIME,
    resol_g=g_resol,
    trans_only=TRANS_ONLY
)
loader_train = DataLoader(ds_train, batch_size=500,shuffle=True)

ds_val = DatasetFeatANDtgy(
    path_feat_by_subj=os.path.join(DIR_RUNTIME_DATA,'df_features_val.csv'),
    path_state_history_max_grade=os.path.join(DIR_RUNTIME_DATA,'df_state_history_sampled_max_val.csv'),
    max_grade=MAX_STATES, 
    max_time=N_TIME,
    resol_g=g_resol,
    trans_only=TRANS_ONLY
)
loader_val = DataLoader(ds_val, batch_size=1000)

ds_test = DatasetFeatANDtgy(
    path_feat_by_subj=os.path.join(DIR_RUNTIME_DATA,'df_features_test.csv'),
    path_state_history_max_grade=os.path.join(DIR_RUNTIME_DATA,'df_state_history_sampled_max_test.csv'),
    max_grade=MAX_STATES, 
    max_time=N_TIME,
    resol_g=g_resol,
    trans_only=TRANS_ONLY
)
loader_test = DataLoader(ds_test, batch_size=1000)

In [ ]:
len(ds_train)

In [ ]:
len(ds_val)

## Specify model

In [ ]:
if USE_GPU:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
else:
    device = 'cpu'
device

## Load model

In [ ]:
dir_logs = os.path.join(DIR_RUNTIME_RESULTS, 'lightning_logs')
dir_logs

In [ ]:
latest_ver = sorted(os.listdir(dir_logs))[-1]
latest_ver = 'version_3'

In [ ]:
checkpoint_path = [i for i in os.listdir(os.path.join(dir_logs, f'{latest_ver}/checkpoints')) if i.endswith('ckpt')][-1]
checkpoint_path = os.path.join(dir_logs, f'{latest_ver}/checkpoints/{checkpoint_path}')
checkpoint_path

In [ ]:
import json
dir_checkpoint = DIR_RUNTIME_RESULTS


# Opening JSON file
with open(os.path.join(dir_checkpoint,"best_hyper_params.json"), 'r') as openfile:
 
    # Reading from json file
    best_hyper_params = json.load(openfile)
best_hyper_params

In [ ]:
from monotonic_nn_surv_surf.utils.surv_surf_latent import SurvSurfLatent, LatentFeatFC
from monotonic_nn_surv_surf.utils.pl_model_wrapper import LitSurvSurf

In [ ]:
n_feat_neurons = best_hyper_params['n_feat_neurons']
n_monoton_neurons = best_hyper_params['n_monoton_neurons']
n_monotone_layers = best_hyper_params['n_monotone_layers']
n_feat_layers = best_hyper_params['n_feat_layers']
p_dropout = best_hyper_params['p_dropout']
learning_rate = best_hyper_params['learning_rate']

model = SurvSurfLatent(
    mono_net_sizes=[n_feat_neurons] + [n_monoton_neurons]*n_monotone_layers + [1],
    latent_feat_transformer=LatentFeatFC(
        input_size=3, 
        output_size=n_feat_neurons, 
        neurons_per_layer=(n_feat_layers-1)*[n_feat_neurons],
        dropout_p=p_dropout
    ),
)
from monotonic_nn_surv_surf.utils import losses

model_lit = LitSurvSurf(model=model, loss_fn=losses.loss_dydg, lr=learning_rate, print_epoch=True)

model_lit_loaded = model_lit.load_from_checkpoint(checkpoint_path)
model_loaded_core = model_lit_loaded.model
model_loaded_core = model_loaded_core.to(device)
model_loaded_core.eval()

## Training-set prediction wirte-out

In [ ]:
pred = []
truth = []
ts_all = []
gs_all = []
with torch.no_grad():
    for i, tdata in enumerate(loader_train):
        xs, ts, gs, ys, weights = tdata
        xs, ts, gs, ys = xs.to(device), ts.to(device), gs.to(device), ys.to(device)
        gs.requires_grad_()
        ts.requires_grad_()
        toutputs = model_loaded_core(ts=ts, gs=gs, xs=xs)
        ts_all += list(ts.cpu().numpy()[:,0])
        gs_all += list(gs.cpu().numpy()[:,0])
        truth += list(ys.cpu().numpy()[:,0])
        pred += list(toutputs.cpu().numpy()[:,0])

pred = np.array(pred)
truth = np.array(truth)
ts_all = np.array(ts_all)
gs_all = np.array(gs_all)


df_train_results = ds_train.observed.copy()
df_train_results['pred'] = pred
df_train_results['truth'] = truth
df_train_results['ts'] = (ts_all*N_TIME).round(0)
df_train_results['gs'] = (gs_all*MAX_STATES).round(0)
df_train_results.to_csv(os.path.join(DIR_RESUlTS_END,'df_train_results.csv'))

del pred
del truth
del ts_all
del gs_all

In [ ]:
prop_true_all = df_train_results['truth'].mean()
prop_true_all

In [ ]:
df_train_results['pred'].hist()

In [ ]:
bins = df_train_results.groupby('truth').apply(
    lambda x: x['pred'].sort_values().iloc[::x['pred'].size//11]
).values
bins = sorted(np.unique(bins))
if bins[-1] < 0.9:
    bins = np.r_[bins, [0.9]]
bins

In [ ]:
df_cal = []
for lower, upper in zip(bins, bins[1:]):
    if upper == 1:
        upper = 1.01
    selector_in_bin = (
        (df_train_results['pred'] >= lower) &
        (df_train_results['pred'] < upper)
    )
    df_in_bin = df_train_results.loc[
        selector_in_bin,
        :
    ].copy()
    
    middle = df_in_bin['pred'].mean()
    prop_positive = df_in_bin['truth'].mean()
    prop_negative = 1-prop_positive

    prop_positive_scaled = prop_positive/prop_true_all
    prop_negative_scaled = prop_negative/(1-prop_true_all)
    prop_positive_adj = prop_positive_scaled/(prop_positive_scaled + prop_negative_scaled)

    df_cal.append(
        {
            'pred_prob':middle,
            'prop_truth_pos_adj':prop_positive
        }
    )
df_cal.append(
    {
        'pred_prob':1,
        'prop_truth_pos_adj':1
    }
)
df_cal = pd.DataFrame(df_cal).sort_values('pred_prob')

In [ ]:
df_cal

In [ ]:

def clean_non_monotonic_points(df):
    df_copy = df.copy()
    df_naless = df_copy.loc[df_copy['prop_truth_pos_adj'].notna(),:]
    diff = df_naless['prop_truth_pos_adj'].diff()
    diff.iloc[0] = 0
    if all(diff >= 0):
        return df_naless
    else:
        selector = (diff >= 0)
        idx_keep = diff.index[
            selector
        ]
        return clean_non_monotonic_points(df_naless.loc[idx_keep,:])
df_cal_monotone = clean_non_monotonic_points(df_cal)
df_cal_monotone

In [ ]:
from scipy.stats import linregress
from numpy import interp
x = df_cal_monotone['pred_prob']
y = df_cal_monotone['prop_truth_pos_adj']

# y = np.log(y/(1-y))
selector = ~y.isna()
x = x[selector]
y = y[selector]
#calib = linregress(x=x, y=y)

# def get_calib_pred(pred):
#     linear_cal = pred*calib.slope + calib.intercept
#     return np.exp(linear_cal)/(1+np.exp(linear_cal))

def get_calib_pred(pred):
    return interp(pred, xp=x, fp=y)

#calib

In [ ]:
df_plot = df_cal.copy()
df_plot['pred_prob_cal'] = get_calib_pred(df_plot['pred_prob'])
sns.scatterplot(
    df_plot,
    x='pred_prob',
    y='prop_truth_pos_adj'
)
sns.scatterplot(
    df_plot,
    x='pred_prob_cal',
    y='prop_truth_pos_adj'
)

## Val-set performance

In [ ]:
pred = []
truth = []
ts_all = []
gs_all = []
with torch.no_grad():
    for i, tdata in enumerate(loader_val):
        xs, ts, gs, ys, weights = tdata
        xs, ts, gs, ys = xs.to(device), ts.to(device), gs.to(device), ys.to(device)
        toutputs = model_loaded_core(ts=ts, gs=gs, xs=xs)
        ts_all += list(ts.cpu().numpy()[:,0])
        gs_all += list(gs.cpu().numpy()[:,0])
        truth += list(ys.cpu().numpy()[:,0])
        pred += list(toutputs.cpu().numpy()[:,0])

pred = np.array(pred)
truth = np.array(truth)
ts_all = np.array(ts_all)
gs_all = np.array(gs_all)


df_val_results = ds_val.observed.copy()
df_val_results['pred'] = pred
df_val_results['truth'] = truth
df_val_results['ts'] = (ts_all*N_TIME).round(0)
df_val_results['gs'] = (gs_all*MAX_STATES).round(0)
df_val_results.head()

del pred
del truth
del ts_all
del gs_all

In [ ]:
df_val_results.head()

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score
fpr, tpr, thresholds = roc_curve(y_true=df_val_results['truth'], y_score=df_val_results['pred'])
print('ROC-AUC = {}'.format(roc_auc_score(y_true=df_val_results['truth'], y_score=df_val_results['pred'])))
import matplotlib.pyplot as plt
idx = np.argmax((1-fpr) + tpr)
thresh_opt = thresholds[idx]
print('optimal thresh: {}'.format(thresh_opt))
plt.plot(fpr, tpr)
plt.xlabel('FPR')
plt.ylabel('TPR')

In [ ]:
conf_mat_all = pd.crosstab(
    df_val_results['pred'] > thresh_opt,
    df_val_results['truth']
)
conf_mat_all

In [ ]:
conf_mat_by_subj = conf_mat_all.copy()
conf_mat_by_subj.loc[:,:] = 0
subj_nan_in_conf_mat = []
n_subj = 0
for subj, df in df_val_results.groupby('subject'):
    conf_mat = pd.crosstab(
        df['pred'] > thresh_opt,
        df['truth']
    )
    if conf_mat.shape != (2,2):
        subj_nan_in_conf_mat.append(subj)
    else:
        n_subj +=1
        conf_mat_by_subj += conf_mat
conf_mat_by_subj = conf_mat_by_subj/n_subj
conf_mat_by_subj

In [ ]:
len(subj_nan_in_conf_mat)

In [ ]:
from sklearn.metrics import balanced_accuracy_score
balanced_accuracy_score(y_true=df_val_results['truth'], y_pred=df_val_results['pred'] > thresh_opt)

In [ ]:
from sklearn.metrics import accuracy_score
df_test_imbalance_by_index = df_val_results.groupby(['ts','gs']).apply(
    lambda df: df['y'].mean()
)

df = df_test_imbalance_by_index.reset_index().pivot(columns='ts', index='gs', values=0)
df.index = df.index.astype(int)
df.columns = df.columns.astype(int)
fig, ax = plt.subplots(1,1, figsize=(15, 3))
sns.heatmap(
    data=df.astype(float),
    ax=ax,
    cbar_kws={'label': 'prop +ve'},
    annot=True
    
)
ax.set(xlabel='ts', ylabel='gs')
######################################
df_test_size_by_index = df_val_results.groupby(['ts','gs']).apply(
    lambda df: df.shape[0]
)

df = df_test_size_by_index.reset_index().pivot(columns='ts', index='gs', values=0)
df.index = df.index.astype(int)
df.columns = df.columns.astype(int)
fig, ax = plt.subplots(1,1, figsize=(15, 3))
sns.heatmap(
    data=df.astype(float),
    ax=ax,
    cbar_kws={'label': 'sample_size'},
    annot=False
    
)
ax.set(xlabel='ts', ylabel='gs')
######################################

df_test_perform_by_index= df_val_results.groupby(['ts','gs']).apply(
    lambda df: accuracy_score(y_true=df['truth'], y_pred=df['pred'] > thresh_opt)
)

df = df_test_perform_by_index.reset_index().pivot(columns='ts', index='gs', values=0)
df.index = df.index.astype(int)
df.columns = df.columns.astype(int)
fig, ax = plt.subplots(1,1, figsize=(15, 3))
sns.heatmap(
    data=df.astype(float),
    ax=ax,
    vmin=0,
    vmax=1,
    cbar_kws={'label': 'accuracy'},
    annot=True
    
)
ax.set(xlabel='ts', ylabel='gs')

## Test set performance

In [ ]:
pred = []
truth = []
ts_all = []
gs_all = []
with torch.no_grad():
    for i, tdata in enumerate(loader_test):
        xs, ts, gs, ys, weights = tdata
        xs, ts, gs, ys = xs.to(device), ts.to(device), gs.to(device), ys.to(device)
        toutputs = model_loaded_core(ts=ts, gs=gs, xs=xs)
        ts_all += list(ts.cpu().numpy()[:,0])
        gs_all += list(gs.cpu().numpy()[:,0])
        truth += list(ys.cpu().numpy()[:,0])
        pred += list(toutputs.cpu().numpy()[:,0])

pred = np.array(pred)
truth = np.array(truth)
ts_all = np.array(ts_all)
gs_all = np.array(gs_all)


df_test_results = ds_test.observed.copy()
df_test_results['pred'] = pred
df_test_results['truth'] = truth
df_test_results['ts'] = (ts_all*N_TIME).round(0)
df_test_results['gs'] = (gs_all*MAX_STATES).round(0)
df_test_results.head()
df_test_results.to_csv(os.path.join(DIR_RESUlTS_END,'df_test_results.csv'))
del pred
del truth
del ts_all
del gs_all


In [ ]:
conf_mat_all = pd.crosstab(
    df_test_results['pred'] > thresh_opt,
    df_test_results['truth']
)
conf_mat_all


In [ ]:
conf_mat_by_subj = conf_mat_all.copy()
conf_mat_by_subj.loc[:,:] = 0
subj_nan_in_conf_mat = []
n_subj = 0
for subj, df in df_test_results.groupby('subject'):
    conf_mat = pd.crosstab(
        df['pred'] > thresh_opt,
        df['truth']
    )
    if conf_mat.shape != (2,2):
        subj_nan_in_conf_mat.append(subj)
    else:
        n_subj +=1
        conf_mat_by_subj += conf_mat
conf_mat_by_subj = conf_mat_by_subj/n_subj
conf_mat_by_subj


In [ ]:
len(subj_nan_in_conf_mat)


In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score
fpr, tpr, thresholds = roc_curve(y_true=df_test_results['truth'], y_score=df_test_results['pred'])
print('ROC-AUC = {}'.format(roc_auc_score(y_true=df_test_results['truth'], y_score=df_test_results['pred'])))
import matplotlib.pyplot as plt
print('optimal thresh: {}'.format(thresh_opt))
plt.plot(fpr, tpr)
plt.xlabel('FPR')
plt.ylabel('TPR')


In [ ]:
from sklearn.metrics import precision_recall_curve, f1_score
from numpy import trapz
precision, recall, thresholds = precision_recall_curve(y_true=df_test_results['truth'], probas_pred=df_test_results['pred'])

print('PR-AUC = {}'.format(trapz(y=precision[::-1], x=recall[::-1])))
import matplotlib.pyplot as plt
# print('optimal thresh: {}'.format(thresh_opt))
plt.plot(recall, precision)
plt.xlabel('recall')
plt.ylabel('precision')

In [ ]:
pr = precision + recall
idx_ = np.where(pr == pr.max())[0][0]
thresh_max_pr = thresholds[idx_]
thresh_max_pr, precision[idx_], recall[idx_]

In [ ]:
from sklearn.metrics import balanced_accuracy_score
balanced_accuracy_score(y_true=df_test_results['truth'], y_pred=df_test_results['pred'] > thresh_opt)

In [ ]:

from sklearn.metrics import accuracy_score
df_test_imbalance_by_index = df_test_results.groupby(['ts','gs']).apply(
    lambda df: df['y'].mean()
)

df = df_test_imbalance_by_index.reset_index().pivot(columns='ts', index='gs', values=0)
df.index = df.index.astype(int)
df.columns = df.columns.astype(int)
fig, ax = plt.subplots(1,1, figsize=(15, 3))
sns.heatmap(
    data=df.astype(float),
    ax=ax,
    cbar_kws={'label': 'prop +ve'},
    annot=True
    
)
ax.set(xlabel='ts', ylabel='gs')
######################################
df_test_size_by_index = df_val_results.groupby(['ts','gs']).apply(
    lambda df: df.shape[0]
)

df = df_test_size_by_index.reset_index().pivot(columns='ts', index='gs', values=0)
df.index = df.index.astype(int)
df.columns = df.columns.astype(int)
fig, ax = plt.subplots(1,1, figsize=(15, 3))
sns.heatmap(
    data=df.astype(float),
    ax=ax,
    cbar_kws={'label': 'sample_size'},
    annot=False
    
)
ax.set(xlabel='ts', ylabel='gs')
######################################

df_test_perform_by_index= df_test_results.groupby(['ts','gs']).apply(
    lambda df: accuracy_score(y_true=df['truth'], y_pred=df['pred'] > thresh_opt)
)

df = df_test_perform_by_index.reset_index().pivot(columns='ts', index='gs', values=0)
df.index = df.index.astype(int)
df.columns = df.columns.astype(int)
fig, ax = plt.subplots(1,1, figsize=(15, 3))
sns.heatmap(
    data=df.astype(float),
    ax=ax,
    vmin=0,
    vmax=1,
    cbar_kws={'label': 'accuracy'},
    annot=True
    
)
ax.set(xlabel='ts', ylabel='gs')

## Performance by surface

In [ ]:
def get_cliff_score(df):
    df_sorted = df.sort_values('g')
    log_truth = np.log(df_sorted['truth'])
    diff = log_truth.diff().iloc[1:]
    if any(pd.isna(diff)):
        df_sorted['cliff_score'] = np.nan
    else:
        mean = np.median(diff)
        new = diff.copy()
        new = np.exp(new - mean)
        df_sorted['cliff_score'] = np.r_[[0], new.values]
    return df_sorted

In [ ]:
def pred_all_gs_ts_one_subj(x, max_time, max_states):
    ts = [i for i in range(max_time+1)]
    gs = [i+1 for i in range(max_states)]

    ts_long = ts*len(gs)
    gs_long = []
    for g in gs:
        gs_long += [g]* len(ts)

    ts_long = np.array(ts_long)[:,None]
    gs_long = np.array(gs_long)[:,None]
    xs_long = np.array([x]*gs_long.shape[0])
    xs = torch.tensor(xs_long, dtype=torch.float32)
    xs = xs.to(device)


    with torch.no_grad():
        t_tensor = torch.tensor(ts_long/N_TIME, dtype=torch.float32)
        t_tensor = t_tensor.to(device)
        g_tensor = torch.tensor(gs_long/MAX_STATES, dtype=torch.float32)
        g_tensor = g_tensor.to(device)
        prob = model_loaded_core(ts=t_tensor, gs=g_tensor, xs=xs)

    df_surf_long = pd.DataFrame(
        {
            'predicted': prob.detach().cpu().numpy().squeeze(),
            't': ts_long.squeeze(),
            'g': gs_long.squeeze(),
        }
    )
    return df_surf_long


def get_pred_vs_theory_surv_surfs(df_features_by_subj, df_surfs_theory_long):
    df_surfs_pred_long = []
    for idx in df_features_by_subj.index[::]:
        subj = df_features_by_subj.loc[idx, 'subject']
        
        x = df_features_by_subj.loc[idx, ['0','1','2']].values
        df_surf_long = pred_all_gs_ts_one_subj(x=x, max_time=N_TIME, max_states=MAX_STATES)
        df_surf_long['subj'] = subj
        df_surfs_pred_long.append(df_surf_long)


    df_surfs_pred_long = pd.concat(df_surfs_pred_long)

    df_surfs_pred_vs_theory = df_surfs_pred_long.merge(
        right=df_surfs_theory_long,
        on=['subj','g','t'],
        how='left'
    ).dropna()
    df_surfs_pred_vs_theory['residual'] = (
        df_surfs_pred_vs_theory['predicted']
        - df_surfs_pred_vs_theory['truth']
    ).abs()
    df_surfs_pred_vs_theory['percent_residual'] = df_surfs_pred_vs_theory['residual']/df_surfs_pred_vs_theory['truth']
    return df_surfs_pred_vs_theory

In [ ]:
df_surfs_theory = pd.read_csv(os.path.join(DIR_INPUTS,'df_surfs.csv'), index_col=0)
df_surfs_theory.head()

In [ ]:
df_surfs_theory_long = df_surfs_theory.melt(
    id_vars=['subj','g'],
    value_vars=[str(i) for i in range(12)],
    value_name='truth',
    var_name='t'
)
df_surfs_theory_long['t'] = df_surfs_theory_long['t'].astype(int)
df_surfs_theory_long = df_surfs_theory_long.loc[df_surfs_theory_long['g']!= 0,:]

### Train-set surface

In [ ]:
title='train'

In [ ]:
df_features_train = pd.read_csv(os.path.join(DIR_RUNTIME_DATA,'df_features_train.csv'), index_col=0)
df_features_train.head()

In [ ]:
df_surfs_pred_vs_theory = get_pred_vs_theory_surv_surfs(
    df_features_train, 
    df_surfs_theory_long
)

In [ ]:
df_surfs_pred_vs_theory = df_surfs_pred_vs_theory.groupby(['subj','t']).apply(get_cliff_score)
df_surfs_pred_vs_theory['predicted_calib'] = get_calib_pred(df_surfs_pred_vs_theory['predicted'])
df_surfs_pred_vs_theory['residual_calib'] = (df_surfs_pred_vs_theory['truth'] - df_surfs_pred_vs_theory['predicted_calib']).abs()

In [ ]:
df_surfs_pred_vs_theory['residual'].describe()

In [ ]:
for col in ['predicted', 'truth','percent_residual','predicted_calib','residual','residual_calib']:
    df_test_perform_by_index= df_surfs_pred_vs_theory.groupby(['g','t'])[col].mean()

    df = df_test_perform_by_index.reset_index().pivot(columns='t', index='g', values=col)
    df.index = df.index.astype(int)
    df.columns = df.columns.astype(int)
    fig, ax = plt.subplots(1,1, figsize=(15, 3))
    sns.heatmap(
        data=df.astype(float),
        ax=ax,
        vmin=0,
        vmax=1,
        cbar_kws={'label': col},
        annot=True
        
    )
    ax.set(xlabel='t', ylabel='g', title=title)

In [ ]:
df_surfs_pred_vs_theory.isna().any()

In [ ]:
for col in [
    'residual',
    'residual_calib'
]:
    fig,ax = plt.subplots(1,1,)
    df_surfs_pred_vs_theory[col].hist(bins=20, ax=ax)
    ax.set(xlabel=col, ylabel='number of t,g combination', title=title)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes = axes.flatten()

ax = axes[0]
sns.scatterplot(
    data=df_surfs_pred_vs_theory,
    x='cliff_score',
    y='residual',
    alpha=0.3,
    hue='g',
    s=5,
    ax=ax
)
ax.set(xscale='log')
ax = axes[1]
sns.scatterplot(
    data=df_surfs_pred_vs_theory,
    x='cliff_score',
    y='residual',
    alpha=0.3,
    hue='t',
    s=5,
    ax=ax
)
ax.set(xscale='log')
fig.suptitle(title)

In [ ]:
fig, axes = plt.subplots(2,2, figsize=(12, 8))
axes = axes.flatten()
ax = axes[0]
sns.scatterplot(
    data=df_surfs_pred_vs_theory,
    x='predicted',
    y='truth',
    alpha=0.3,
    s=5,
    hue='t',
    ax=ax
)
ax.plot([0,1], [0,1], ls='--', color='grey')

ax = axes[1]
sns.scatterplot(
    data=df_surfs_pred_vs_theory,
    x='predicted',
    y='truth',
    alpha=0.3,
    s=5,
    hue='g',
    ax=ax
)
ax.plot([0,1], [0,1], ls='--', color='grey')


ax = axes[2]
sns.scatterplot(
    data=df_surfs_pred_vs_theory,
    x='predicted_calib',
    y='truth',
    alpha=0.3,
    s=5,
    hue='t',
    ax=ax
)
ax.plot([0,1], [0,1], ls='--', color='grey')

ax = axes[3]
sns.scatterplot(
    data=df_surfs_pred_vs_theory,
    x='predicted_calib',
    y='truth',
    alpha=0.3,
    s=5,
    hue='g',
    ax=ax
)
ax.plot([0,1], [0,1], ls='--', color='grey')

fig.suptitle(title)


### Val-set surface

In [ ]:
title = 'validation'

In [ ]:
df_features_val = pd.read_csv(os.path.join(DIR_RUNTIME_DATA,'df_features_val.csv'), index_col=0)
df_features_val.head()

In [ ]:
df_surfs_pred_vs_theory = get_pred_vs_theory_surv_surfs(
    df_features_val, 
    df_surfs_theory_long
)

In [ ]:
df_surfs_pred_vs_theory = df_surfs_pred_vs_theory.groupby(['subj','t']).apply(get_cliff_score)
df_surfs_pred_vs_theory['predicted_calib'] = get_calib_pred(df_surfs_pred_vs_theory['predicted'])
df_surfs_pred_vs_theory['residual_calib'] = (df_surfs_pred_vs_theory['truth'] - df_surfs_pred_vs_theory['predicted_calib']).abs()

In [ ]:
df_surfs_pred_vs_theory['residual'].describe()

In [ ]:
for col in ['predicted', 'truth','percent_residual','predicted_calib','residual','residual_calib']:
    df_test_perform_by_index= df_surfs_pred_vs_theory.groupby(['g','t'])[col].mean()

    df = df_test_perform_by_index.reset_index().pivot(columns='t', index='g', values=col)
    df.index = df.index.astype(int)
    df.columns = df.columns.astype(int)
    fig, ax = plt.subplots(1,1, figsize=(15, 3))
    sns.heatmap(
        data=df.astype(float),
        ax=ax,
        vmin=0,
        vmax=1,
        cbar_kws={'label': col},
        annot=True
        
    )
    ax.set(xlabel='t', ylabel='g', title=title)

In [ ]:
df_surfs_pred_vs_theory.isna().any()

In [ ]:
for col in [
    'residual',
    'residual_calib'
]:
    fig,ax = plt.subplots(1,1,)
    df_surfs_pred_vs_theory[col].hist(bins=20, ax=ax)
    ax.set(xlabel=col, ylabel='number of t,g combination', title=title)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes = axes.flatten()

ax = axes[0]
sns.scatterplot(
    data=df_surfs_pred_vs_theory,
    x='cliff_score',
    y='residual',
    alpha=0.3,
    hue='g',
    s=5,
    ax=ax
)
ax.set(xscale='log')
ax = axes[1]
sns.scatterplot(
    data=df_surfs_pred_vs_theory,
    x='cliff_score',
    y='residual',
    alpha=0.3,
    hue='t',
    s=5,
    ax=ax
)
ax.set(xscale='log')
fig.suptitle(title)

In [ ]:
fig, axes = plt.subplots(2,2, figsize=(12, 8))
axes = axes.flatten()
ax = axes[0]
sns.scatterplot(
    data=df_surfs_pred_vs_theory,
    x='predicted',
    y='truth',
    alpha=0.3,
    s=5,
    hue='t',
    ax=ax
)
ax.plot([0,1], [0,1], ls='--', color='grey')

ax = axes[1]
sns.scatterplot(
    data=df_surfs_pred_vs_theory,
    x='predicted',
    y='truth',
    alpha=0.3,
    s=5,
    hue='g',
    ax=ax
)
ax.plot([0,1], [0,1], ls='--', color='grey')


ax = axes[2]
sns.scatterplot(
    data=df_surfs_pred_vs_theory,
    x='predicted_calib',
    y='truth',
    alpha=0.3,
    s=5,
    hue='t',
    ax=ax
)
ax.plot([0,1], [0,1], ls='--', color='grey')

ax = axes[3]
sns.scatterplot(
    data=df_surfs_pred_vs_theory,
    x='predicted_calib',
    y='truth',
    alpha=0.3,
    s=5,
    hue='g',
    ax=ax
)
ax.plot([0,1], [0,1], ls='--', color='grey')

fig.suptitle(title)


### Test-set surface

In [ ]:
title='test'

In [ ]:
df_features_test = pd.read_csv(os.path.join(DIR_RUNTIME_DATA,'df_features_test.csv'), index_col=0)
df_features_test

In [ ]:
df_state_history_sampled_test =  pd.read_csv(os.path.join(DIR_RUNTIME_DATA,'df_state_history_sampled_max_test.csv'), index_col=0)
df_state_history_sampled_test.head()

In [ ]:
df_surfs_pred_vs_theory = get_pred_vs_theory_surv_surfs(
    df_features_test, 
    df_surfs_theory_long
)

In [ ]:
df_surfs_pred_vs_theory = df_surfs_pred_vs_theory.groupby(['subj','t']).apply(get_cliff_score)
df_surfs_pred_vs_theory['predicted_calib'] = get_calib_pred(df_surfs_pred_vs_theory['predicted'])
df_surfs_pred_vs_theory['residual_calib'] = (df_surfs_pred_vs_theory['truth'] - df_surfs_pred_vs_theory['predicted_calib']).abs()

In [ ]:
df_surfs_pred_vs_theory['residual'].describe()

In [ ]:
for col in ['predicted', 'truth','percent_residual','predicted_calib','residual','residual_calib']:
    df_test_perform_by_index= df_surfs_pred_vs_theory.groupby(['g','t'])[col].mean()

    df = df_test_perform_by_index.reset_index().pivot(columns='t', index='g', values=col)
    df.index = df.index.astype(int)
    df.columns = df.columns.astype(int)
    fig, ax = plt.subplots(1,1, figsize=(15, 3))
    sns.heatmap(
        data=df.astype(float),
        ax=ax,
        vmin=0,
        vmax=1,
        cbar_kws={'label': col},
        annot=True
        
    )
    ax.set(xlabel='t', ylabel='g', title=title)

In [ ]:
df_surfs_pred_vs_theory.isna().any()

In [ ]:
for col in [
    'residual',
    'residual_calib'
]:
    fig,ax = plt.subplots(1,1,)
    df_surfs_pred_vs_theory[col].hist(bins=20, ax=ax)
    ax.set(xlabel=col, ylabel='number of t,g combination', title=title)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes = axes.flatten()

ax = axes[0]
sns.scatterplot(
    data=df_surfs_pred_vs_theory,
    x='cliff_score',
    y='residual',
    alpha=0.3,
    hue='g',
    s=5,
    ax=ax
)
ax.set(xscale='log')
ax = axes[1]
sns.scatterplot(
    data=df_surfs_pred_vs_theory,
    x='cliff_score',
    y='residual',
    alpha=0.3,
    hue='t',
    s=5,
    ax=ax
)
ax.set(xscale='log')
fig.suptitle(title)

In [ ]:
fig, axes = plt.subplots(2,2, figsize=(12, 8))
axes = axes.flatten()
ax = axes[0]
sns.scatterplot(
    data=df_surfs_pred_vs_theory,
    x='predicted',
    y='truth',
    alpha=0.3,
    s=5,
    hue='t',
    ax=ax
)
ax.plot([0,1], [0,1], ls='--', color='grey')

ax = axes[1]
sns.scatterplot(
    data=df_surfs_pred_vs_theory,
    x='predicted',
    y='truth',
    alpha=0.3,
    s=5,
    hue='g',
    ax=ax
)
ax.plot([0,1], [0,1], ls='--', color='grey')


ax = axes[2]
sns.scatterplot(
    data=df_surfs_pred_vs_theory,
    x='predicted_calib',
    y='truth',
    alpha=0.3,
    s=5,
    hue='t',
    ax=ax
)
ax.plot([0,1], [0,1], ls='--', color='grey')

ax = axes[3]
sns.scatterplot(
    data=df_surfs_pred_vs_theory,
    x='predicted_calib',
    y='truth',
    alpha=0.3,
    s=5,
    hue='g',
    ax=ax
)
ax.plot([0,1], [0,1], ls='--', color='grey')

fig.suptitle(title)


### Test-set tragectory

In [ ]:
thresh_opt_calib = get_calib_pred(thresh_opt)

In [ ]:
import seaborn as sns

for subj in df_features_test.loc[:, 'subject'].iloc[::df_features_test.shape[0]//50]:
    selector = df_surfs_pred_vs_theory['subj'] == subj
    df_surf_truth_subj = df_surfs_pred_vs_theory.loc[selector,:].pivot(values='truth', index='g', columns='t')

    df_surf = df_surfs_pred_vs_theory.loc[
        df_surfs_pred_vs_theory['subj'] == subj,:
    ].pivot(values='predicted', index='g', columns='t')
    pred_trace = df_surf.apply(lambda x: np.max(np.r_[np.where(x>thresh_opt_calib)[0]+1,[0]]), axis=0)

    fig, ax = plt.subplots(1,1, figsize=(15, 3))
    sns.heatmap(
        data=df_surf,
        ax=ax,
        vmin=0,
        vmax=1,
        cbar_kws={'label': 'P(reaching max grade by time)'},
        annot=True
        
    )
    ax.set(xlabel='time', ylabel='max grade', title=f'subj={subj} PREDICTED')
    

    df_surf = df_surfs_pred_vs_theory.loc[
        df_surfs_pred_vs_theory['subj'] == subj,:
    ].pivot(values='predicted_calib', index='g', columns='t')
    pred_trace = df_surf.apply(lambda x: np.max(np.r_[np.where(x>thresh_opt_calib)[0]+1,[0]]), axis=0)
    
    fig, ax = plt.subplots(1,1, figsize=(15, 3))
    sns.heatmap(
        data=df_surf,
        ax=ax,
        vmin=0,
        vmax=1,
        cbar_kws={'label': 'P(reaching max grade by time)'},
        annot=True
        
    )
    ax.set(xlabel='time', ylabel='max grade', title=f'subj={subj} PREDICTED (Calib)')

    fig, ax = plt.subplots(1,1, figsize=(15, 3))
    sns.heatmap(
        data=df_surf_truth_subj,
        ax=ax,
        vmin=0,
        vmax=1,
        cbar_kws={'label': 'P(reaching max grade by time)'},
        annot=True
        
    )
    ax.set(xlabel='time', ylabel='max grade', title=f'subj={subj} TRUTH')

    df = df_state_history_sampled_test.loc[df_state_history_sampled_test['subject'] == subj,:]
    fig, ax = plt.subplots(1,1, figsize=(12, 3))
    ax.plot(
        df['time'], df['state'], label='observed grade', marker='*'
    )
    ax.plot(
        df['time'], df['state_max_by_time'], label='observed max grade by time', marker='*'
    )
    ax.plot(
        pred_trace.index.values, pred_trace.values, label='pred max grade by time', marker='*'
    )
    ax.legend()
    ax.set(xlabel='time', ylabel='grade', title=f'subj={subj}', ylim=(0,MAX_STATES+1), xlim=(0,N_TIME))
    del df_surf